In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F


torch.manual_seed(0)
torch.set_printoptions(precision=2, sci_mode=False, linewidth=120)

print("torch",torch.__version__)

torch 2.14.1+cpu


In [4]:
def show(mat, title="", row_labels=None, col_labels=None, decimals=2):
    """Print a 2-D tensor with token names on the rows and columns."""
    row_labels = row_labels or TOKENS
    col_labels = col_labels or TOKENS
    if title:
        print(title)
    print("       " + "".join(f"{c:>8}" for c in col_labels))
    for name, row in zip(row_labels, mat.tolist()):
        body = "".join(f"{val:8.{decimals}f}" for val in row)
        print(f"{name:>6} {body}")
    print()

In [5]:
TOKENS = ["the", "cat", "sat", "on", "mat"]

B = 1
T = len(TOKENS)
d_model = 8
n_head = 2
d_k =  d_model//n_head

x = torch.randn(B, T, d_model)

print("tokens:", TOKENS)
print("x.shape:", tuple(x.shape), " (batch, time, width)")
print()
print("x[0] -- one row per token:")
print(x[0])


tokens: ['the', 'cat', 'sat', 'on', 'mat']
x.shape: (1, 5, 8)  (batch, time, width)

x[0] -- one row per token:
tensor([[-1.13, -1.15, -0.25, -0.43,  0.85,  0.69, -0.32, -2.12],
        [ 0.32, -1.26,  0.35,  0.31,  0.12,  1.24,  1.12, -0.25],
        [-1.35, -1.70,  0.57,  0.79,  0.60, -1.56, -0.34,  1.85],
        [-0.22, -0.74,  0.56,  0.26, -0.17, -0.68,  0.94,  0.49],
        [ 1.20,  0.08, -1.20, -0.00, -0.52, -0.31, -1.58,  1.71]])


In [6]:
show(x[0][:, :5], title="x, 5 first 5 columns")

x, 5 first 5 columns
            the     cat     sat      on     mat
   the    -1.13   -1.15   -0.25   -0.43    0.85
   cat     0.32   -1.26    0.35    0.31    0.12
   sat    -1.35   -1.70    0.57    0.79    0.60
    on    -0.22   -0.74    0.56    0.26   -0.17
   mat     1.20    0.08   -1.20   -0.00   -0.52



In [7]:
col_labels = [f"d{i}" for i in range(5)]
show(x[0][:,:5],"x, first 5 columns", col_labels=col_labels)

x, first 5 columns
             d0      d1      d2      d3      d4
   the    -1.13   -1.15   -0.25   -0.43    0.85
   cat     0.32   -1.26    0.35    0.31    0.12
   sat    -1.35   -1.70    0.57    0.79    0.60
    on    -0.22   -0.74    0.56    0.26   -0.17
   mat     1.20    0.08   -1.20   -0.00   -0.52



In [8]:
naive_score = x[0] @ x[0].T
print("x[0].shape:", tuple(x[0].shape))
print("naive_scores.shape:", tuple(naive_score.shape))
show(naive_score,"naive score")

x[0].shape: (5, 8)
naive_scores.shape: (5, 5)
naive score
            the     cat     sat      on     mat
   the     8.62    2.00   -1.39   -1.10   -4.91
   cat     2.00    4.77   -0.54    1.21   -2.77
   sat    -1.39   -0.54   11.98    3.61    1.41
    on    -1.10    1.21    3.61    2.59   -1.35
   mat    -4.91   -2.77    1.41   -1.35    8.67



In [9]:
W_q = nn.Linear(d_model, d_model, bias=False)
W_k = nn.Linear(d_model, d_model, bias=False)
W_v = nn.Linear(d_model, d_model, bias=False)
W_o = nn.Linear(d_model, d_model, bias=False)
q = W_q(x)
k = W_k(x)
v = W_v(x)

print("x.shape:",tuple(x.shape))
print("q.shape:", tuple(q.shape))
print("k.shape:", tuple(k.shape))
print("v.shape:", tuple(v.shape))

x.shape: (1, 5, 8)
q.shape: (1, 5, 8)
k.shape: (1, 5, 8)
v.shape: (1, 5, 8)


In [10]:
scores = q @ k.transpose(-2,-1)
print("scores.shape:", tuple(scores.shape))
print()
show(scores[0], "scores= q @K^T")

scores.shape: (1, 5, 5)

scores= q @K^T
            the     cat     sat      on     mat
   the     0.70   -1.33    0.45   -0.22    0.33
   cat    -0.69   -0.17   -1.03   -0.20    0.78
   sat    -0.51   -0.31   -0.34   -0.35    0.07
    on    -0.45    0.47   -0.55   -0.03   -0.27
   mat    -0.87   -0.58   -0.46   -0.53    1.74



In [11]:
def row(t, style="{:6.2f}"):
    return "["+" ".join(style.format(v) for v in t.tolist())+"]"

small = torch.tensor([0.5, 1.2, 0.8])
big = small *16

print("scores", row(small), "-> softmax", row(F.softmax(small, dim=-1), "{:6.1%}"))
print("scores", row(big),   "-> softmax", row(F.softmax(big,   dim=-1), "{:6.1%}"))


scores [  0.50   1.20   0.80] -> softmax [ 22.9%  46.1%  30.9%]
scores [  8.00  19.20  12.80] -> softmax [  0.0%  99.8%   0.2%]


In [12]:
scaled = scores / d_k ** 0.5

print("std of raw scores   :", round(scores.std().item(), 3))
print("std of scaled scores:", round(scaled.std().item(), 3))
print()
show(scaled[0], "scaled = scores / sqrt(d_k)")

std of raw scores   : 0.648
std of scaled scores: 0.324

scaled = scores / sqrt(d_k)
            the     cat     sat      on     mat
   the     0.35   -0.67    0.22   -0.11    0.16
   cat    -0.34   -0.09   -0.51   -0.10    0.39
   sat    -0.26   -0.15   -0.17   -0.18    0.04
    on    -0.23    0.24   -0.27   -0.01   -0.14
   mat    -0.44   -0.29   -0.23   -0.26    0.87



In [13]:
max_seq_len = 16

sent = torch.ones(max_seq_len, max_seq_len, dtype=torch.bool)
sent_mask = torch.tril(sent)

print("sent.shape:", tuple(sent.shape))
print("mask.shape:", tuple(sent_mask.shape))

show(sent[:T, :T].int())
print()
show(sent_mask[:T, :T].int())

sent.shape: (16, 16)
mask.shape: (16, 16)
            the     cat     sat      on     mat
   the     1.00    1.00    1.00    1.00    1.00
   cat     1.00    1.00    1.00    1.00    1.00
   sat     1.00    1.00    1.00    1.00    1.00
    on     1.00    1.00    1.00    1.00    1.00
   mat     1.00    1.00    1.00    1.00    1.00


            the     cat     sat      on     mat
   the     1.00    0.00    0.00    0.00    0.00
   cat     1.00    1.00    0.00    0.00    0.00
   sat     1.00    1.00    1.00    0.00    0.00
    on     1.00    1.00    1.00    1.00    0.00
   mat     1.00    1.00    1.00    1.00    1.00



In [14]:
mask = torch.tril(
    torch.ones(T, T, dtype=torch.bool)
)

show(mask)
masked = scaled.masked_fill(~mask, float("-inf"))

show(masked[0], "scaled")

            the     cat     sat      on     mat
   the     1.00    0.00    0.00    0.00    0.00
   cat     1.00    1.00    0.00    0.00    0.00
   sat     1.00    1.00    1.00    0.00    0.00
    on     1.00    1.00    1.00    1.00    0.00
   mat     1.00    1.00    1.00    1.00    1.00

scaled
            the     cat     sat      on     mat
   the     0.35    -inf    -inf    -inf    -inf
   cat    -0.34   -0.09    -inf    -inf    -inf
   sat    -0.26   -0.15   -0.17    -inf    -inf
    on    -0.23    0.24   -0.27   -0.01    -inf
   mat    -0.44   -0.29   -0.23   -0.26    0.87



In [15]:
weights = F.softmax(masked, dim=-1)

out = weights @ v
print("weights.shape:",tuple(weights.shape))
print("out.shape:", tuple(out.shape))
print("v.shape:", tuple(v.shape))

show(weights[0],"attention weights")


weights.shape: (1, 5, 5)
out.shape: (1, 5, 8)
v.shape: (1, 5, 8)
attention weights
            the     cat     sat      on     mat
   the     1.00    0.00    0.00    0.00    0.00
   cat     0.44    0.56    0.00    0.00    0.00
   sat     0.31    0.35    0.34    0.00    0.00
    on     0.21    0.33    0.20    0.26    0.00
   mat     0.12    0.14    0.15    0.14    0.45



# Checkpoint

In [16]:
def single_head_attention(x, W_q, W_k, W_v, mask):
    
    B, T, d_model =  x.shape
   
    q = W_q(x)
    k = W_k(x)
    v = W_v(x)
    
    scores = q @ k.transpose(-2, -1)
    scores = scores / d_model**0.5
    
    causal = mask[:T, :T]
    scores = scores.masked_fill(~causal, float("-inf"))
    
    weights = F.softmax(scores, dim=-1)
    # print("weights:",weights.shape)
    out = weights @ v
    
    return  out, weights

In [17]:
h_out, h_weights= single_head_attention(x, W_q, W_k, W_v, mask    )
print("out:",h_out.shape)
print("x:", x.shape)


out: torch.Size([1, 5, 8])
x: torch.Size([1, 5, 8])


In [18]:
head_a, head_b = None, None

torch.manual_seed(101)
head_a = [
    nn.Linear(d_model, d_model, bias=False)
    for _ in range(3)
]
torch.manual_seed(202)

head_b = [
    nn.Linear(d_model, d_model, bias=False)
    for _ in range(3)
]

o_a, w_a= single_head_attention(x,*head_a, mask)
o_b, w_b= single_head_attention(x,*head_b, mask)

show(w_a[0],"head A")
show(w_b[0], "head B")

head A
            the     cat     sat      on     mat
   the     1.00    0.00    0.00    0.00    0.00
   cat     0.48    0.52    0.00    0.00    0.00
   sat     0.44    0.38    0.18    0.00    0.00
    on     0.28    0.25    0.23    0.24    0.00
   mat     0.35    0.25    0.07    0.15    0.17

head B
            the     cat     sat      on     mat
   the     1.00    0.00    0.00    0.00    0.00
   cat     0.46    0.54    0.00    0.00    0.00
   sat     0.55    0.31    0.14    0.00    0.00
    on     0.26    0.27    0.22    0.25    0.00
   mat     0.25    0.12    0.08    0.12    0.42



In [19]:
# projected = W_q(x)
# print("1. after W_q            :", tuple(projected.shape))

# split = projected.view(B, T, n_head, d_k)
# print("2. after .view          :", tuple(split.shape), " (B, T, n_heads, d_k)")

# q_heads = split.transpose(1, 2)
# print("3. after .transpose(1,2):", tuple(q_heads.shape), " (B, n_heads, T, d_k)")

In [20]:
q_heads = W_q(x).view(B, T, n_head, d_k).transpose(1,2)
print("q_heads:", tuple(q_heads.shape), "(B, n_heads, T, d_k)")

q_heads: (1, 2, 5, 4) (B, n_heads, T, d_k)


# Multihead Attention Checkpoint

In [21]:
def multi_head_attention(x, W_q, W_k, W_v, W_o, mask, n_heads):
    
    B, T, d_model = x.shape
    d_k = d_model // n_heads
    
    q = W_q(x).view(B, T, n_heads, d_k).transpose(1,2)
    k = W_k(x).view(B, T, n_heads, d_k).transpose(1,2)
    v = W_v(x).view(B, T, n_heads, d_k).transpose(1,2)
    
    scores = q @ k.transpose(-2, -1)
    scores = scores / d_model**0.5
    
    # causal = mask[:T, :T]
    scores = scores.masked_fill(~mask[:T, :T], float("-inf"))
    
    weights = F.softmax(scores, dim=-1)
    # print("weights:",weights.shape)
    out = weights @ v
    
    out = out.transpose(1,2).contiguous().view(B, T, d_model)
    
    return W_o(out), weights

In [22]:
mha_out, mha_w = multi_head_attention(x, W_q, W_k, W_v, W_o, mask, n_head)


print("in :", tuple(x.shape))
print("out:", tuple(mha_out.shape))
print("weights:", tuple(mha_w.shape), " (B, n_heads, T, T)")

in : (1, 5, 8)
out: (1, 5, 8)
weights: (1, 2, 5, 5)  (B, n_heads, T, T)


# Final Class MultiHead Attention

In [23]:
class MyMultiHeadAttention(nn.Module):
    """Every token looks at every earlier token, through several lenses at once.

    Input:  (B, T, d_model)
    Output: (B, T, d_model)
    """

    def __init__(self, config):
        super().__init__()

        self.n_heads = config.n_heads
        self.d_k = config.d_k

        # One full-width projection per role, carved into heads in forward().
        self.W_q = nn.Linear(config.d_model, config.d_model, bias=False)
        self.W_k = nn.Linear(config.d_model, config.d_model, bias=False)
        self.W_v = nn.Linear(config.d_model, config.d_model, bias=False)

        self.W_o = nn.Linear(config.d_model, config.d_model)
        mask = torch.tril(
            torch.ones(config.max_seq_len, config.max_seq_len, dtype=torch.bool)
        )

        self.register_buffer("mask", mask, persistent=False)

    def forward(self, x):

        batch_size, seq_len, d_model = x.shape

        q = self.W_q(x).view(batch_size, seq_len, self.n_heads, self.d_k).transpose(1, 2)
        k = self.W_k(x).view(batch_size, seq_len, self.n_heads, self.d_k).transpose(1, 2) # => (B, n_heads, T, d_k)
        v = self.W_v(x).view(batch_size, seq_len, self.n_heads, self.d_k).transpose(1, 2)
        
        
        
        scores = q @ k.transpose(-2, -1)
        scores = scores / d_k**0.5
        scores = scores.masked_fill(~self.mask, float("-inf"))

        weights = F.softmax(scores, dim=-1)
        out = weights @ v

        out = out.transpose(1, 2).contiguous().view(batch_size, seq_len, d_model)
        # .transpose(1, 2) ===> (B, T, n_heads, d_k)
        # .view(B, T, d_model) ===> (B, T, d_model)

        return W_o(out), weights

# Kv Cache

In [26]:
def row(t, style="{:6.2f}"):
    return "[" + " ".join(style.format(v) for v in t.tolist()) + "]"

for step in range(1, T +1):
    mha_out, _ = multi_head_attention(x[:, :step, :], W_q, W_k, W_v, W_o, mask, n_head)
    prefix = " ".join(TOKENS[:step])
    print(f"step:{step}")
    print(f"prefix input:{prefix}")
    print(f"output:{row(mha_out[0,0,:])}")


step:1
prefix input:the
output:[  0.47   0.25   0.12   0.96  -0.02   0.31  -0.46   0.03]
step:2
prefix input:the cat
output:[  0.47   0.25   0.12   0.96  -0.02   0.31  -0.46   0.03]
step:3
prefix input:the cat sat
output:[  0.47   0.25   0.12   0.96  -0.02   0.31  -0.46   0.03]
step:4
prefix input:the cat sat on
output:[  0.47   0.25   0.12   0.96  -0.02   0.31  -0.46   0.03]
step:5
prefix input:the cat sat on mat
output:[  0.47   0.25   0.12   0.96  -0.02   0.31  -0.46   0.03]


In [35]:
Tokens = ["the", "cat", "sat", "on", "mat"]

past_length = 3
new_length = 1

past = x[:, :3, :] # "the" , "cat" , "sat"
new = x[:, 3:4, :] # "on"

past_k = W_k(past).view(B, past_length, n_head, d_k).transpose(1, 2)  # (B, n_head, 3, d_k)
past_v = W_v(past).view(B, past_length, n_head, d_k).transpose(1, 2)  # (B, n_head, 3, d_k)

kv_cache = (past_k, past_v)


q = W_q(new).view(B, new_length, n_head, d_k).transpose(1,2)
k = W_k(new).view(B, new_length, n_head, d_k).transpose(1,2)
v = W_v(new).view(B, new_length, n_head, d_k).transpose(1,2)


past_k, past_v = kv_cache
past_len = past_k.size(2)
seq_length = new.shape[1]


k = torch.cat([past_k, k], dim=2)
v = torch.cat([past_v, v], dim=2)


total_length = past_length + seq_length


print("past_len :", past_length, " tokens already in the cache")
print("seq_len :", seq_length, " tokens fed in this step")
print("total_len:", total_length, " tokens attention can see")
print()

print("q:", tuple(q.shape), " <- 1 query   (q for 'the', 'cat', 'sat' is NOT rebuilt)")
print("k:", tuple(k.shape), " <- 4 keys     (3 reused from cache + 1 new)")
print("v:", tuple(v.shape), " <- 4 values   (3 reused from cache + 1 new)")


past_len : 3  tokens already in the cache
seq_len : 1  tokens fed in this step
total_len: 4  tokens attention can see

q: (1, 2, 1, 4)  <- 1 query   (q for 'the', 'cat', 'sat' is NOT rebuilt)
k: (1, 2, 4, 4)  <- 4 keys     (3 reused from cache + 1 new)
v: (1, 2, 4, 4)  <- 4 values   (3 reused from cache + 1 new)


In [31]:
past_k.shape, new.shape

(torch.Size([1, 3, 2, 4]), torch.Size([1, 1, 8]))